# Starting with the V2 -  
### New features and everything , along with betchmark(logisticR)

## 2. Imports + MAC safety line:

In [3]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import pandas as pd
import numpy as np

pd.set_option('display.max_columns',None)
print("IMPORTS READY")

IMPORTS READY


## 2. Load RAW files, confirm shapes:

In [5]:
results_df   = pd.read_csv('../data/raw/results.csv')
goals_df     = pd.read_csv('../data/raw/goalscorers.csv')
shootouts_df = pd.read_csv('../data/raw/shootouts.csv')
names_df     = pd.read_csv('../data/raw/former_names.csv')
wc2026_df    = pd.read_csv('../data/wc2026/wc2026_results.csv')

for name, df in [('results', results_df), ('goalscorers', goals_df),
                  ('shootouts', shootouts_df), ('former_names', names_df),
                  ('wc2026_results', wc2026_df)]:
    print(f"{name:<16} shape: {df.shape}")

results          shape: (49477, 9)
goalscorers      shape: (47601, 8)
shootouts        shape: (678, 5)
former_names     shape: (36, 4)
wc2026_results   shape: (94, 9)


## 3. Merge Historical + WC2026 Results:

In [6]:
print("results_df columns: ",list(results_df.columns))
print("wc2026_df columns: ",list(wc2026_df.columns))

results_df columns:  ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
wc2026_df columns:  ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


## 4. combine + Convert Dates:

In [8]:
wc2026_aligned = wc2026_df[results_df.columns.tolist()]

combined_df = pd.concat([results_df,wc2026_aligned],ignore_index=True)
combined_df['date']= pd.to_datetime(combined_df['date'])

print(f"Historical matches: {len(results_df)}")
print(f"WC2026 matches: {len(wc2026_aligned)}")
print(f"COmbined total: {len(combined_df)}")
print(f"Date range : {combined_df['date'].min()} , {combined_df['date'].max()}")

dupes = combined_df.duplicated(subset = ['date','home_team','away_team'],keep=False)
print(f"potential duplicate matches : {dupes.sum()}")
if dupes.sum()>0: print(combined_df[dupes][['date','home_team','away_team','home_score','away_score']])

Historical matches: 49477
WC2026 matches: 94
COmbined total: 49571
Date range : 1872-11-30 00:00:00 , 2026-07-06 00:00:00
potential duplicate matches : 132
            date  home_team       away_team  home_score  away_score
9641  1974-02-17     Tahiti   New Caledonia         2.0         1.0
9642  1974-02-17     Tahiti   New Caledonia         1.0         2.0
49352 2026-06-06  Gibraltar  Cayman Islands         4.0         1.0
49361 2026-06-06  Gibraltar  Cayman Islands         4.0         1.0
49405 2026-06-11     Mexico    South Africa         NaN         NaN
...          ...        ...             ...         ...         ...
49544 2026-06-17      Ghana          Panama         1.0         0.0
49545 2026-06-23    England           Ghana         0.0         0.0
49546 2026-06-23     Panama         Croatia         0.0         1.0
49547 2026-06-27     Panama         England         0.0         2.0
49548 2026-06-27    Croatia           Ghana         2.0         1.0

[132 rows x 5 columns]


### Sanity : 

In [9]:
# Tag every row so we can categorize the duplicates properly
combined_df['source'] = ['historical'] * len(results_df) + ['wc2026_manual'] * len(wc2026_aligned)

dupe_mask = combined_df.duplicated(subset=['date','home_team','away_team'], keep=False)
dupes_df = combined_df[dupe_mask].sort_values(['date','home_team'])

# Split into: matches involving our WC2026 backfill vs everything else
wc2026_overlap = dupes_df[dupes_df['source'] == 'wc2026_manual']
other_dupes    = dupes_df[~dupes_df.index.isin(wc2026_overlap.index)]
other_dupes    = other_dupes[~other_dupes[['date','home_team','away_team']].apply(tuple, axis=1).isin(
    wc2026_overlap[['date','home_team','away_team']].apply(tuple, axis=1)
)]

print(f"Duplicates overlapping with our WC2026 backfill : {len(wc2026_overlap)}")
print(f"Duplicates unrelated to WC2026 (pre-existing)     : {len(other_dupes)}")
print(f"\nUnrelated duplicates:")
print(other_dupes[['date','home_team','away_team','home_score','away_score','source']])

Duplicates overlapping with our WC2026 backfill : 64
Duplicates unrelated to WC2026 (pre-existing)     : 4

Unrelated duplicates:
            date  home_team       away_team  home_score  away_score  \
9641  1974-02-17     Tahiti   New Caledonia         2.0         1.0   
9642  1974-02-17     Tahiti   New Caledonia         1.0         2.0   
49352 2026-06-06  Gibraltar  Cayman Islands         4.0         1.0   
49361 2026-06-06  Gibraltar  Cayman Islands         4.0         1.0   

           source  
9641   historical  
9642   historical  
49352  historical  
49361  historical  


In [10]:
# Rule 1 — for any match date/team-pair present in BOTH sources,
# trust wc2026_manual (we specifically verified it against real events earlier)
# and drop the historical/results.csv version
overlap_keys = set(wc2026_overlap[['date','home_team','away_team']].apply(tuple, axis=1))

def is_overlap_historical(row):
    return (row['source'] == 'historical' and
            (row['date'], row['home_team'], row['away_team']) in overlap_keys)

combined_df = combined_df[~combined_df.apply(is_overlap_historical, axis=1)].copy()

# Rule 2 — for exact duplicate rows unrelated to WC2026 (identical everything), drop the extra copy
combined_df = combined_df.drop_duplicates(subset=['date','home_team','away_team','home_score','away_score'])

# Rule 3 — remaining conflicting-score duplicates (like Tahiti/New Caledonia) — 
# can't verify which is correct without an external source, and it's 1 pair out of ~49,500 rows.
# Safest move: drop both rather than guess.
still_duped = combined_df.duplicated(subset=['date','home_team','away_team'], keep=False)
print(f"Remaining unresolved duplicates: {still_duped.sum()}")
print(combined_df[still_duped][['date','home_team','away_team','home_score','away_score','source']])

combined_df = combined_df[~still_duped].copy()
combined_df = combined_df.drop(columns=['source'])

print(f"\nFinal combined shape: {combined_df.shape}")

Remaining unresolved duplicates: 2
           date home_team      away_team  home_score  away_score      source
9641 1974-02-17    Tahiti  New Caledonia         2.0         1.0  historical
9642 1974-02-17    Tahiti  New Caledonia         1.0         2.0  historical

Final combined shape: (49504, 9)


## update the current data on our side:

In [13]:
import pandas as pd

wc2026_df = pd.read_csv('../data/wc2026/wc2026_results.csv', parse_dates=['date'])
print(f"Before: {len(wc2026_df)} rows, max date {wc2026_df['date'].max().date()}")

missing_r16 = pd.DataFrame([
    {'date': '2026-07-07', 'home_team': 'Argentina', 'away_team': 'Egypt',
     'home_score': 3, 'away_score': 2, 'tournament': 'FIFA World Cup',
     'city': 'Atlanta', 'country': 'United States', 'neutral': True},
    {'date': '2026-07-07', 'home_team': 'Switzerland', 'away_team': 'Colombia',
     'home_score': 0, 'away_score': 0, 'tournament': 'FIFA World Cup',
     'city': 'Vancouver', 'country': 'Canada', 'neutral': True},
])
missing_r16['date'] = pd.to_datetime(missing_r16['date'])

wc2026_df = pd.concat([wc2026_df, missing_r16], ignore_index=True)
wc2026_df = wc2026_df.drop_duplicates(subset=['date','home_team','away_team'])

print(f"After:  {len(wc2026_df)} rows, max date {wc2026_df['date'].max().date()}")
wc2026_df.to_csv('../data/wc2026/wc2026_new_results.csv', index=False)

Before: 94 rows, max date 2026-07-06
After:  96 rows, max date 2026-07-07


In [14]:
shootouts_df = pd.read_csv('../data/raw/shootouts.csv', parse_dates=['date'])

new_shootouts = pd.DataFrame([
    {'date': '2026-06-29', 'home_team': 'Paraguay', 'away_team': 'Germany',
     'winner': 'Paraguay', 'first_shooter': 'Germany'},
    {'date': '2026-06-29', 'home_team': 'Morocco', 'away_team': 'Netherlands',
     'winner': 'Morocco', 'first_shooter': 'Netherlands'},
    {'date': '2026-06-29', 'home_team': 'Egypt', 'away_team': 'Australia',
     'winner': 'Egypt', 'first_shooter': 'Australia'},
    {'date': '2026-07-07', 'home_team': 'Switzerland', 'away_team': 'Colombia',
     'winner': 'Switzerland', 'first_shooter': 'Colombia'},
])
new_shootouts['date'] = pd.to_datetime(new_shootouts['date'])

shootouts_df = pd.concat([shootouts_df, new_shootouts], ignore_index=True)
shootouts_df = shootouts_df.drop_duplicates(subset=['date','home_team','away_team'])

print(f"shootouts.csv: now {len(shootouts_df)} rows, max date {shootouts_df['date'].max().date()}")
shootouts_df.to_csv('../data/raw/shootouts_new.csv', index=False)

shootouts.csv: now 682 rows, max date 2026-07-07
